# Clase 21 — PCA + K-Means + t-SNE: Reduccion de dimensiones y segmentacion

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador x UDLA

---

**Objetivos de hoy:**
1. Entender **por que** necesitamos PCA cuando tenemos muchas variables.
2. Aplicar PCA **paso a paso**: escalar, ajustar, interpretar componentes.
3. Combinar **PCA + K-Means** para segmentar paises por felicidad.
4. Interpretar clusters con **profiling** y visualizaciones.
5. Conocer **t-SNE** como alternativa no lineal para visualizacion.

**Dataset:** World Happiness Report — 153 paises, 12 columnas.

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE

print("Imports OK")

---
## 1. Cargar y explorar los datos

In [ ]:
URL = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-21/world_happiness.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

In [ ]:
df.describe()

153 paises, 12 columnas. 7 indicadores numericos que usaremos para PCA.

In [ ]:
features = ["Economy", "Family", "Health", "Freedom", "Generosity", "Corruption", "Job Satisfaction"]
df_clean = df.dropna(subset=features).copy()
print(f"Paises despues de limpiar NaN: {len(df_clean)}")

In [ ]:
fig = px.scatter(
    df_clean, x="Economy", y="Health", color="Region",
    hover_data=["Country", "Happiness Score"],
    title="Economy vs Health por Region",
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Paises con mayor economia tienden a tener mejor salud. Pero esto es solo 2 de 7 variables. PCA nos permitira ver las 7 al mismo tiempo en un solo grafico.

---
## 2. La maldicion de la dimensionalidad

Con 7 variables necesitariamos **21 scatter plots** (combinaciones de 7 tomadas de 2 en 2). Cada uno muestra solo 2 de 7 dimensiones — una vista parcial.

**PCA (Principal Component Analysis)** comprime las 7 variables en 2-3 "super-variables" que capturan la **mayor variacion posible**. Es como tomar una foto 2D de un objeto 7D desde el mejor angulo.

**Necesitamos PCA.**

---
## 3. PCA paso a paso

### 3.1 Escalar (obligatorio)

In [ ]:
X = df_clean[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Antes de escalar:")
print(X.describe().loc[["mean", "std"]].round(2))
print("\nDespues de escalar:")
print(pd.DataFrame(X_scaled, columns=features).describe().loc[["mean", "std"]].round(2))

**StandardScaler es obligatorio para PCA** (misma razon que K-Means: trabaja con distancias y varianzas). Sin escalar, las variables con valores mas grandes dominarian los componentes.

### 3.2 Ajustar PCA (todos los componentes)

In [ ]:
pca = PCA()  # todos los componentes primero, para ver el scree plot
X_pca = pca.fit_transform(X_scaled)

print(f"Componentes: {pca.n_components_}")
print(f"Varianza explicada: {pca.explained_variance_ratio_.round(3)}")

### 3.3 Scree plot — Cuantos componentes necesitamos?

In [ ]:
var_exp = pca.explained_variance_ratio_
cum_var = np.cumsum(var_exp)

fig = px.bar(
    x=[f"PC{i+1}" for i in range(len(var_exp))], y=var_exp,
    title="Varianza explicada por componente",
    labels={"x": "Componente", "y": "Varianza explicada"},
    color_discrete_sequence=["#C82B40"])
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
fig = px.line(
    x=list(range(1, 8)), y=cum_var, markers=True,
    title="Varianza acumulada",
    labels={"x": "Numero de componentes", "y": "Varianza acumulada"})
fig.add_hline(y=0.7, line_dash="dash", annotation_text="70%", line_color="#16A34A")
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%", line_color="#EA580C")
fig.update_layout(template="plotly_white")
fig.show()

print(f"PC1: {var_exp[0]:.1%}")
print(f"PC1+PC2: {cum_var[1]:.1%}")
print(f"PC1+PC2+PC3: {cum_var[2]:.1%}")

**Interpretacion:** 2 componentes capturan ~71% de la varianza. Podemos representar 151 paises en un solo scatter 2D y conservar el 71% de la informacion de las 7 variables originales.

### 3.4 Que significan los componentes? — Loadings

In [ ]:
loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=features,
    columns=["PC1", "PC2", "PC3"])
print(loadings.round(3))

In [ ]:
fig = px.imshow(
    loadings.values,
    x=["PC1", "PC2", "PC3"],
    y=features,
    color_continuous_scale="RdBu_r", zmin=-0.6, zmax=0.6,
    title="Loadings: que significa cada componente",
    text_auto=".2f")
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion de loadings:**
- **PC1:** Economy, Health, Job Satisfaction y Family tienen loadings positivos altos -> PC1 representa **"desarrollo y calidad de vida"**.
- **PC2:** Generosity, Corruption y Freedom dominan -> PC2 representa **"libertad social y confianza"**.

El heatmap muestra de un vistazo que variables "pesan" en cada componente. Rojo = loading positivo alto, azul = negativo.

### 3.5 Visualizar paises en el espacio PCA

In [ ]:
df_clean["PC1"] = X_pca[:, 0]
df_clean["PC2"] = X_pca[:, 1]

fig = px.scatter(
    df_clean, x="PC1", y="PC2", color="Region",
    hover_data=["Country", "Happiness Score"],
    title="151 paises en 2D (PCA)",
    labels={"PC1": "PC1 — Desarrollo", "PC2": "PC2 — Libertad/confianza"},
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los paises se agrupan por region de forma natural. Europa Occidental (PC1 alto) vs Africa (PC1 bajo). Latinoamerica ocupa PC1 medio. Noruega aparece arriba-derecha (alto desarrollo). Chad abajo-izquierda. Ecuador en zona media, junto a otros latinoamericanos. PCA revelo esta estructura comprimiendo 7 variables en 2.

### 3.6 Encontrar paises especificos

In [ ]:
for country in ["Ecuador", "Norway", "Chad", "Costa Rica", "Japan"]:
    row = df_clean[df_clean["Country"] == country]
    if len(row) > 0:
        print(f"{country}: PC1={row['PC1'].values[0]:.2f}, PC2={row['PC2'].values[0]:.2f}, Region={row['Region'].values[0]}")

**Interpretacion:**
- **Noruega:** PC1 alto (muy desarrollado) y PC2 positivo (alta libertad/confianza). Arriba-derecha del grafico.
- **Chad:** PC1 muy bajo (bajo desarrollo). Abajo-izquierda.
- **Ecuador:** PC1 medio, cerca de otros paises latinoamericanos. Zona de transicion.
- **Costa Rica:** PC1 medio, PC2 relativamente alto (buena libertad para su nivel de desarrollo).
- **Japan:** PC1 alto (desarrollado) pero PC2 bajo (menor confianza social relativa).

### 3.7 Scatter 3D con 3 componentes

In [ ]:
pca3 = PCA(n_components=3)
X_pca3 = pca3.fit_transform(X_scaled)
df_clean["PC3"] = X_pca3[:, 2]

fig = px.scatter_3d(df_clean, x="PC1", y="PC2", z="PC3", color="Region",
    hover_data=["Country", "Happiness Score"], opacity=0.7,
    title=f"3 componentes ({pca3.explained_variance_ratio_.sum():.1%} varianza)")
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** 3 componentes capturan ~81% de la varianza. Rota el grafico para ver estructura que no era visible en 2D. PC3 agrega informacion adicional (generosity vs family/health).

---
## 4. PCA + K-Means: segmentacion en el espacio reducido

### 4.1 Silhouette para K=2..7

In [ ]:
sil_scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca[:, :2])
    sil_scores.append(silhouette_score(X_pca[:, :2], labels))

fig = px.line(
    x=list(range(2, 8)), y=sil_scores, markers=True,
    title="Silhouette Score por K (espacio PCA)",
    labels={"x": "K", "y": "Silhouette"})
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** K=3 tiene el mejor Silhouette Score. Tres segmentos es una division natural: paises desarrollados, en transicion y en desarrollo.

### 4.2 Ajustar K-Means con K=3

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean["cluster"] = km.fit_predict(X_pca[:, :2])

print(f"Paises por cluster:\n{df_clean['cluster'].value_counts().sort_index()}")

### 4.3 Visualizar clusters

In [ ]:
fig = px.scatter(
    df_clean, x="PC1", y="PC2",
    color=df_clean["cluster"].astype(str),
    hover_data=["Country", "Region", "Happiness Score"],
    title="3 segmentos de paises (PCA + K-Means)",
    labels={"PC1": "PC1 — Desarrollo", "PC2": "PC2 — Libertad"},
    color_discrete_sequence=["#2563EB", "#C82B40", "#16A34A"],
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los 3 clusters separan claramente: paises desarrollados (derecha), en transicion (centro) y en desarrollo (izquierda). K-Means encontro esta estructura sin que le dijeramos nada sobre regiones o niveles de desarrollo.

### 4.4 Coloreado por Happiness Score

In [ ]:
fig = px.scatter(
    df_clean, x="PC1", y="PC2",
    color="Happiness Score", color_continuous_scale="RdYlGn",
    hover_data=["Country", "Region"],
    title="Mismos datos, coloreados por Happiness Score")
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los clusters se alinean con la felicidad: el cluster de paises desarrollados (derecha) tiene los scores mas altos (verde). PCA + K-Means descubrio la estructura **sin nunca ver el Happiness Score**. Eso confirma que los 7 indicadores realmente explican la felicidad.

### 4.5 Usando sklearn Pipeline

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10)),
])
df_clean["cluster_pipe"] = pipeline.fit_predict(X)

print(f"Clusters del pipeline:\n{df_clean['cluster_pipe'].value_counts().sort_index()}")

**Interpretacion:** Pipeline encadena todos los pasos (escalar -> PCA -> K-Means) en un solo objeto. Mismo resultado, codigo mas limpio y sin riesgo de olvidar un paso.

### 4.6 Perfil de clusters

In [ ]:
perfil = df_clean.groupby("cluster")[["Happiness Score"] + features].mean().round(2)
print(perfil)

for c in sorted(df_clean["cluster"].unique()):
    countries = df_clean[df_clean["cluster"] == c]["Country"].tolist()
    avg = df_clean[df_clean["cluster"] == c]["Happiness Score"].mean()
    print(f"\nCluster {c} ({len(countries)} paises, felicidad={avg:.1f}): {countries[:8]}...")

**Interpretacion del perfil:**
- **Cluster con Happiness alto:** Economy, Health, Family y Job Satisfaction altos -> **"Paises desarrollados"** (Noruega, Suiza, etc.).
- **Cluster medio:** Valores intermedios en todo -> **"Paises en transicion"** (Ecuador, Brasil, etc.).
- **Cluster bajo:** Economy y Health bajos -> **"Paises en desarrollo"** (Chad, Burundi, etc.).

PCA + K-Means descubrio grupos con niveles de felicidad muy distintos sin usar el Happiness Score como variable de entrada.

---
## 5. Ejercicio: Aplicar PCA + K-Means a Mall Customers

Usa el dataset de Mall Customers de la clase 19. Aplica PCA para reducir las 3 features (Age, income, spending) a 2 componentes. Luego aplica K-Means con K=5.

1. Carga los datos y escala las features.
2. Aplica PCA (todos los componentes). Cuanta varianza capturan 2 componentes?
3. Aplica PCA con 2 componentes + K-Means con K=5.
4. Visualiza el scatter en el espacio PCA coloreado por cluster.
5. Compara con los resultados de clase 19 (sin PCA). PCA agrega mucho con solo 3 variables?

In [ ]:
# Tu codigo aqui


---
## 6. t-SNE — Cuando PCA no alcanza

**PCA es lineal:** busca las direcciones de maxima varianza en linea recta. Funciona muy bien para datos con estructura lineal, pero puede fallar con datos complejos.

**t-SNE (t-distributed Stochastic Neighbor Embedding)** es **no lineal**: preserva las distancias entre vecinos cercanos, revelando clusters y estructuras que PCA no puede ver.

**Diferencia clave:** t-SNE no tiene `.transform()` — solo `.fit_transform()`. No es parametrico: no puedes aplicarlo a datos nuevos. Es solo para **visualizacion**.

### 6.2 t-SNE en World Happiness

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)
df_clean["tsne1"] = X_tsne[:, 0]
df_clean["tsne2"] = X_tsne[:, 1]

fig = px.scatter(df_clean, x="tsne1", y="tsne2", color="Region",
    hover_data=["Country", "Happiness Score"],
    title="t-SNE: World Happiness", opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** t-SNE separa las regiones de forma mas visual que PCA. Pero los ejes no tienen significado (no son "desarrollo" ni "libertad") y las distancias entre grupos lejanos no son confiables. Solo sirve para visualizar, no para preprocesar.

### 6.3 Imágenes como datos: ¿qué ve la computadora?

Antes de aplicar PCA y t-SNE a imágenes, hay que entender cómo se convierten en datos numéricos.

Una imagen **no es una foto** para la máquina — es una **matriz de números**, donde cada número es la **intensidad** de un pixel.

| Tipo | Forma del array | Rango típico | Ejemplo (foto 100×100) |
|------|-----------------|--------------|-------------------------|
| Grises | `(alto, ancho)` | 0 (negro) a 255 (blanco) | 10,000 números |
| RGB | `(alto, ancho, 3)` | 0–255 por canal | 30,000 números |
| RGBA | `(alto, ancho, 4)` | + canal de transparencia | 40,000 números |

Para usar una imagen en ML hay que **aplanarla**: la matriz 2D se convierte en un **vector 1D** — una sola fila del DataFrame con tantas columnas como pixeles.

#### Cargar una imagen real con matplotlib

`plt.imread()` lee cualquier PNG/JPG y devuelve directamente un array de NumPy. Vamos a cargar una imagen real (el logo de la clase) para ver el proceso completo: **cargar → inspeccionar → convertir a grises → aplanar**.

In [ ]:
import urllib.request, io
from PIL import Image

# Cargar una foto real (puedes reemplazar la URL por cualquier imagen)
URL_IMG = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-21/qr_encuesta.jpg"
img = np.array(Image.open(io.BytesIO(urllib.request.urlopen(URL_IMG).read())))

print(f"Forma del array: {img.shape}")           # (alto, ancho, 3) si es RGB
print(f"Tipo: {img.dtype}, rango: {img.min()}-{img.max()}")
print(f"Total de numeros: {img.size:,}")

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f"Imagen original {img.shape}")
plt.axis("off")
plt.show()

In [ ]:
# Convertir a grises (promedio de R, G, B) y aplanar
gris = img.mean(axis=2) if img.ndim == 3 else img
vector = gris.flatten()

print(f"Matriz de grises: {gris.shape}  ({gris.size} numeros)")
print(f"Vector aplanado: {vector.shape}  (1 fila para ML)")
print(f"Primeros 10 valores: {vector[:10].astype(int)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gris, cmap="gray"); axes[0].set_title(f"Grises: matriz {gris.shape}"); axes[0].axis("off")
axes[1].imshow(vector.reshape(1, -1), cmap="gray", aspect="auto")
axes[1].set_title(f"Aplanada: vector ({vector.size},) — 1 fila del 'DataFrame'")
axes[1].set_yticks([])
plt.tight_layout(); plt.show()

**Interpretación:** la imagen se convierte en una sola fila con `alto × ancho` columnas. Cada columna es un pixel. Cualquier modelo (PCA, K-Means, RF) trabaja con esto **igual que con datos tabulares** — no le importa si las columnas son pixeles o `Edad`/`Ingreso`.

Para construir un dataset de N imágenes: `np.stack([plt.imread(f).flatten() for f in archivos])` → matriz `(N, alto*ancho)`.

### 6.4 El dataset `digits`: MNIST en miniatura

Ahora que entendemos imagen→vector, vamos al caso clásico: dígitos escritos a mano.

`load_digits()` de sklearn es la versión reducida del famoso **MNIST** (Modified National Institute of Standards and Technology, 1998). MNIST tiene 70,000 imágenes de 28×28 pixeles. `digits` es una versión más pequeña: **1,797 imágenes de 8×8** pixeles, intensidades 0–16.

Lo que importa: **64 columnas, una por pixel**. Cada fila es un dígito (0–9) ya aplanado.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_dig = digits.data       # (1797, 64) — ya aplanado
y_dig = digits.target     # (1797,)    — etiqueta 0..9
images = digits.images    # (1797, 8, 8) — version no aplanada

print(f"X_dig.shape  = {X_dig.shape}   <- 1797 imagenes, 64 pixeles cada una")
print(f"y_dig.shape  = {y_dig.shape}")
print(f"images.shape = {images.shape}  <- las mismas, pero en 8x8")
print(f"Rango de pixel: {X_dig.min()}-{X_dig.max()}")
print(f"\nPrimera fila (los 64 numeros del primer digito):")
print(X_dig[0].astype(int))

In [ ]:
# Ver la primera imagen como matriz Y como vector — confirma que son lo mismo
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5),
                         gridspec_kw={"width_ratios": [1, 1.4, 2]})

# Panel 1: la imagen
axes[0].imshow(images[0], cmap="gray_r")
axes[0].set_title(f"1. Imagen 8x8\n(digito = {y_dig[0]})", fontweight="bold")
axes[0].axis("off")

# Panel 2: matriz con los numeros encima
axes[1].imshow(images[0], cmap="gray_r", alpha=0.25)
for i in range(8):
    for j in range(8):
        v = int(images[0, i, j])
        c = "white" if v > 8 else "black"
        axes[1].text(j, i, str(v), ha="center", va="center",
                     fontsize=10, fontweight="bold", color=c)
axes[1].set_title("2. Matriz 8x8 de intensidades", fontweight="bold")
axes[1].set_xticks(range(8)); axes[1].set_yticks(range(8))

# Panel 3: vector aplanado
axes[2].imshow(X_dig[0].reshape(1, -1), cmap="gray_r", aspect="auto")
axes[2].set_title("3. Vector aplanado (.flatten()): 64 numeros = 1 fila", fontweight="bold")
axes[2].set_yticks([]); axes[2].set_xticks([0, 15, 31, 47, 63])
plt.tight_layout(); plt.show()

**Interpretación:** los 3 paneles muestran la misma información en 3 formas. La computadora solo ve la última: una fila con 64 números.

In [ ]:
# Visualizar un ejemplo de cada digito
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for d in range(10):
    idx = np.where(y_dig == d)[0][0]
    ax = axes[d // 5, d % 5]
    ax.imshow(images[idx], cmap="gray_r")
    ax.set_title(f"{d}", fontsize=14, fontweight="bold")
    ax.axis("off")
plt.suptitle("Un ejemplo de cada digito (0-9)", fontweight="bold")
plt.tight_layout(); plt.show()

### 6.5 ¿Por qué la relación entre pixeles es NO lineal?

La identidad de un dígito **no está en un pixel** sino en la **forma** — una relación geométrica entre muchos pixeles:

- Un "0" y un "8" pueden tener la **misma cantidad total de tinta** (mismo `sum(pixeles)`): un modelo lineal sobre la suma no los separa.
- Mover el dígito 1 pixel a la derecha cambia *todos* los valores de columna pero es el **mismo número** — no hay correspondencia lineal entre "pixel (3,4) brillante" y "es un 8".
- Lo que distingue dígitos son **bordes, curvas y círculos cerrados** — combinaciones no lineales de pixeles vecinos.

Demostremos que con 2 pixeles **no se pueden separar** los dígitos:

In [ ]:
# Demostración: 2 pixeles no alcanzan para separar digitos
fig = px.scatter(
    x=X_dig[:, 28], y=X_dig[:, 43],
    color=y_dig.astype(str),
    title="Pixel 28 vs Pixel 43: los digitos se solapan completamente (relacion NO lineal)",
    labels={"x": "Pixel 28 (centro)", "y": "Pixel 43 (abajo-derecha)", "color": "Digito"},
    color_discrete_sequence=["#2563EB","#C82B40","#16A34A","#EA580C","#7C3AED",
                             "#06B6D4","#D946EF","#F59E0B","#6B7280","#1E293B"],
    opacity=0.5)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# La matriz de correlacion entre pixeles tampoco es simple
import seaborn as sns
corr = pd.DataFrame(X_dig).corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            xticklabels=False, yticklabels=False, cbar_kws={"label": "correlacion"})
plt.title("Correlacion entre los 64 pixeles\n(bloques = vecinos correlacionados, pero no lineal)",
          fontweight="bold")
plt.xlabel("pixel j"); plt.ylabel("pixel i")
plt.tight_layout(); plt.show()

**Interpretación:** ningún par de pixeles separa los dígitos linealmente. La matriz de correlación tiene **bloques** y **patrones**, no la diagonal limpia que pediría un método puramente lineal. Por eso PCA pierde estructura aquí, y t-SNE (no lineal) la captura.

### 6.6 PCA vs t-SNE en dígitos: la prueba

Aplicamos los dos métodos sobre las mismas 64 dimensiones y comparamos.

> **Atención:** t-SNE en 1797 puntos toma ~10–30 segundos. PCA es instantáneo.

In [ ]:
# Escalar y aplicar PCA en 2D
scaler_dig = StandardScaler()
X_dig_scaled = scaler_dig.fit_transform(X_dig)

pca_dig = PCA(n_components=2)
X_dig_pca = pca_dig.fit_transform(X_dig_scaled)
print(f"PCA: varianza explicada (PC1+PC2) = {pca_dig.explained_variance_ratio_.sum():.1%}")

# t-SNE en 2D (toma ~10-30 seg con 1797 puntos)
tsne_dig = TSNE(n_components=2, random_state=42, perplexity=30, init="pca")
X_dig_tsne = tsne_dig.fit_transform(X_dig_scaled)
print(f"t-SNE: shape = {X_dig_tsne.shape}")

In [ ]:
# Comparacion lado a lado
DIGIT_COLORS = ["#2563EB","#C82B40","#16A34A","#EA580C","#7C3AED",
                "#06B6D4","#D946EF","#F59E0B","#6B7280","#1E293B"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for d in range(10):
    m = y_dig == d
    axes[0].scatter(X_dig_pca[m, 0], X_dig_pca[m, 1],
                    c=DIGIT_COLORS[d], s=10, alpha=0.6, label=str(d))
    axes[1].scatter(X_dig_tsne[m, 0], X_dig_tsne[m, 1],
                    c=DIGIT_COLORS[d], s=10, alpha=0.6, label=str(d))

axes[0].set_title(f"PCA 2D — digitos solapados ({pca_dig.explained_variance_ratio_.sum():.0%} varianza)",
                  fontweight="bold", color="#C82B40")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend(title="Digito", fontsize=8, ncol=5, loc="upper right")

axes[1].set_title("t-SNE 2D — 10 islas claras",
                  fontweight="bold", color="#16A34A")
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")
axes[1].legend(title="Digito", fontsize=8, ncol=5, loc="upper right")

plt.suptitle("64 dimensiones a 2D: PCA pierde, t-SNE revela",
             fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

**Interpretación:** PCA en 2D solo captura ~21% de la varianza — los 10 dígitos se mezclan. t-SNE encuentra **10 islas claras**, una por dígito, sin haber visto las etiquetas. La estructura existe en las 64 dimensiones; PCA (lineal) no llega y t-SNE (no lineal) la revela.

### 6.7 K-Means sobre el espacio reducido

Aplicamos K-Means con K=10 (sabemos que hay 10 dígitos) sobre PCA y sobre t-SNE, y vemos cuál separa mejor.

In [ ]:
km_pca  = KMeans(n_clusters=10, random_state=42, n_init=10).fit(X_dig_pca)
km_tsne = KMeans(n_clusters=10, random_state=42, n_init=10).fit(X_dig_tsne)

sil_pca  = silhouette_score(X_dig_pca,  km_pca.labels_)
sil_tsne = silhouette_score(X_dig_tsne, km_tsne.labels_)

print(f"Silhouette en espacio PCA  : {sil_pca:.3f}")
print(f"Silhouette en espacio t-SNE: {sil_tsne:.3f}")
print(f"\nt-SNE da clusters {'mas' if sil_tsne > sil_pca else 'menos'} compactos.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for c in range(10):
    m_p = km_pca.labels_  == c
    m_t = km_tsne.labels_ == c
    axes[0].scatter(X_dig_pca[m_p, 0],  X_dig_pca[m_p, 1],
                    c=DIGIT_COLORS[c], s=10, alpha=0.6, label=f"C{c}")
    axes[1].scatter(X_dig_tsne[m_t, 0], X_dig_tsne[m_t, 1],
                    c=DIGIT_COLORS[c], s=10, alpha=0.6, label=f"C{c}")
axes[0].set_title(f"K-Means sobre PCA  (silhouette = {sil_pca:.3f})", fontweight="bold")
axes[1].set_title(f"K-Means sobre t-SNE (silhouette = {sil_tsne:.3f})", fontweight="bold")
for a in axes: a.legend(fontsize=7, ncol=5, loc="upper right")
plt.tight_layout(); plt.show()

**Lección práctica:**
- En espacio **PCA** (lineal), K-Means encuentra clusters pero se mezclan — refleja el solapamiento de la proyección.
- En espacio **t-SNE** (no lineal), K-Means recupera casi perfectamente los 10 dígitos.
- **Pero ojo:** no usamos t-SNE para preprocesar en producción (no tiene `.transform()`, no se reproduce con datos nuevos). Lo usamos para **explorar y visualizar**.

### 6.8 Cuándo usar cada uno

| | **PCA** | **t-SNE** |
|---|---|---|
| Tipo | Lineal | No lineal |
| Preserva | Estructura global (distancias grandes) | Vecindarios (puntos cercanos) |
| Paramétrico | Sí: tiene `.transform()` para datos nuevos | **No**: solo `.fit_transform()` |
| Varianza explicada | Sí, sabemos cuánto captura cada PC | No, no tiene métrica de varianza |
| Reproducible | Sí, mismo resultado siempre | Depende de `random_state` y `perplexity` |
| Velocidad | Rápido (incluso 100K filas) | Lento (minutos con >10K filas) |
| Mejor para | Preprocesamiento, reducir ruido | **Visualizar** alta dimensión |

**Regla de bolsillo:** PCA = navaja suiza (rápido, interpretable, paramétrico). t-SNE = microscopio (revela detalles que PCA no ve, pero no sirve para preprocesar).

---
## Resumen de la clase

| Tema | Concepto clave |
|---|---|
| **El problema** | 7 variables = 21 scatter plots. Imposible visualizar todo |
| **PCA** | Comprime N variables en 2-3 "super-variables" que capturan la mayor varianza |
| **StandardScaler** | Obligatorio antes de PCA (trabaja con varianzas) |
| **Scree plot** | Muestra cuantos componentes necesitamos (buscar el codo) |
| **Loadings** | Dicen que significa cada componente (que variables pesan mas) |
| **PCA + K-Means** | PCA reduce dimensiones, K-Means segmenta en el espacio reducido |
| **Pipeline** | Encadena Scaler + PCA + K-Means en un solo objeto |
| **Profiling** | Calcular promedios por cluster para interpretar los segmentos |
| **t-SNE** | Reduccion no lineal — mejor para visualizar, no para preprocesar |
| **PCA vs t-SNE** | PCA = lineal, rapido, parametrico. t-SNE = no lineal, lento, solo visualizacion |